In [1]:
from pynq.overlays.base import BaseOverlay
import time
from datetime import datetime
base = BaseOverlay("base.bit")

In [2]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODA
int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODA
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}


//function to reset pins!
int reset_pmodb(){
    
    for(int i = 5; i < 8; i++)
    {
       write_gpio(i,0); 
    }  
    return 0;
}

Duty Cycle Experimentation and Frquency Testing

In [3]:
import time

blue_pin = 5
green_pin = 6
red_pin = 7

frequency = 60 # 60 Hz frequency shows no noticable blink 
duty_cycle = 50

period = 1 / frequency
time_high = period * duty_cycle / 100
time_low = period - time_high

end_time = time.monotonic() + 30 # 30 second run time!

pin = green_pin

try:
    while time.monotonic() < end_time:
        write_gpio(pin,1)
        time.sleep(time_high)
        write_gpio(pin,0)
        time.sleep(time_low)
finally:
    write_gpio(red_pin,1)
    #reset_pmodb()


Async IO and Button Logic

In [4]:
import asyncio
cond = True
start = True

blue_pin = 5
green_pin = 6
red_pin = 7

pin = red_pin

frequency = 1 # 60 Hz frequency shows no noticable blink 
duty_cycle = 50

period = 1 / frequency
time_high = period * duty_cycle / 100
time_low = period - time_high

async def blink_led():
    global start, pin, time_high,time_low
    while start:
        write_gpio(pin,1)
        await asyncio.sleep(time_high)
        write_gpio(pin,0)
        await asyncio.sleep(time_low)
    

async def get_btns(_loop):
    global start,pin
    while start:
        await asyncio.sleep(0.01)
        button = btns.read()
        if button & 0x1:
            reset_pmodb()
            pin = 7
        elif button & 0x2:
            reset_pmodb()
            pin = 6
        elif button & 0x4:
            reset_pmodb()
            pin = 5
        elif button & 0x8:
            reset_pmodb()
            _loop.stop()
            
          #  cond = False
reset_pmodb()      
btns = base.btns_gpio
loop = asyncio.new_event_loop()
loop.create_task(get_btns(loop))
loop.create_task(blink_led())
loop.run_forever()
loop.close()        
print("Done.")

Done.
